In [ ]:
# Cell 1: check the GPU, then install marker-pdf
import subprocess
# GPU info (first 400 chars of nvidia-smi)
print(subprocess.run(["nvidia-smi"],capture_output=True,text=True).stdout[:400])
r=subprocess.run("pip install -q 'marker-pdf==1.10.2' 2>&1 | tail -4", shell=True, capture_output=True, text=True)
print(r.stdout[-800:]); print("install rc",r.returncode)


In [ ]:
# Cell 2: convert every corpus PDF to markdown with marker

import os, glob, subprocess, shutil
# Kaggle input dataset and working output directory
IN="/kaggle/input/slr-corpus-pdfs"
OUT="/kaggle/working/out"; os.makedirs(OUT, exist_ok=True)
pdfs=sorted(glob.glob("/kaggle/input/**/*.pdf", recursive=True))
print("PDFs:", len(pdfs))

# Counters for success and failure
ok=0; err=[]

# Loop over every PDF and run marker_single
for i,f in enumerate(pdfs,1):
    base=os.path.splitext(os.path.basename(f))[0]
    d=os.path.join(OUT,base); os.makedirs(d, exist_ok=True)
    r=subprocess.run(["marker_single", f, "--output_dir", d], capture_output=True, text=True)
    print(f"[{i}/{len(pdfs)}] {base} rc={r.returncode}", flush=True)
    # Record failures (with a stderr tail) for the final summary
    if r.returncode!=0:
        err.append((base,(r.stderr or "")[-300:]))
    else:
        ok+=1
    # remove large debug files if any
    for dbg in glob.glob(os.path.join(d,"**","*.json"), recursive=True):
        pass

# Summary of results
print("OK:",ok,"ERR:",len(err))
for e in err[:10]: print("ERR:",e[0],e[1])

# Zip the output folder for download
shutil.make_archive("/kaggle/working/slr_full_out","zip",OUT)
print("FILES:", os.listdir("/kaggle/working"))
print("SIZE:", subprocess.run("du -sh /kaggle/working/out",shell=True,capture_output=True,text=True).stdout)
